## Model Performance Discussion

Although the initial model achieved high accuracy, deeper analysis revealed a severe class imbalance issue.

Accuracy alone was misleading.

By applying **class weighting**, the model significantly improved its ability to detect depression cases:

- Recall improved → Better detection of at-risk students
- F1-Score improved → Balanced performance

This highlights the importance of selecting appropriate evaluation metrics in sensitive classification tasks.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("/kaggle/input/student-depression-and-lifestyle-100k-data/student_lifestyle_100k.csv")

df = df.drop("Student_ID", axis=1)

df = pd.get_dummies(df, columns=["Gender", "Department"], drop_first=True)

df.head()

In [ ]:
X = df.drop("Depression", axis=1)
y = df["Depression"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3,random_state=42,stratify=y)

In [ ]:
rf = RandomForestClassifier(n_estimators=100,class_weight='balanced',n_jobs=-1,random_state=42)
rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
importances = rf.feature_importances_
features = X.columns

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=importance_df.head(10), x="Importance", y="Feature")
plt.title("Top 10 Important Features")
plt.show()

In [ ]:
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}
grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1
)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_

In [ ]:
final_pred = best_model.predict(X_test)
print(classification_report(y_test, final_pred))
